# Final models for the submission

**The question:** what exactly will the Stage 12 submission run?

Every model so far was trained for cross-validation, holding slides out. The
submission needs models that have seen every training slide. Decided on
2026-11-19: both heads go forward, both with soft labels; each is trained on all
10,615 packed slides once per seed (0 to 4), and the five models' predicted
probabilities are averaged at prediction time.

| Head | Model (`src/mil.py`) | Loss | Cross-validation (notebook 17): mixed, Radboud to Karolinska, Karolinska to Radboud |
|---|---|---|---|
| `abmil` | `GatedABMIL` | soft labels | 0.886, 0.696, 0.789 |
| `roformer` | `TileTransformerABMIL(rope=True)` | soft labels | 0.887, 0.750, 0.794 |

Those scores are per-seed means; the averaged ensemble was not cross-validated,
and training on every slide gives no held-out score here. The external number
comes from the submission.

Each head's five models are saved together in one file
(`data/models/<head>_soft_labels.pt`, not tracked; `head_path` in
`src/paths.py`), with the feature-set version, settings and commit that made
them. They are uploaded to Kaggle as a dataset for the submission notebook.

## Preamble

In [1]:
"""Make the project's src/ folder importable.

Locally it sits one folder up from notebooks/. On Kaggle it arrives as an
attached dataset, so every attached dataset is checked for a src/ folder, and
the notebook stops with a clear message if there is none.
"""

import glob
import os
import sys

SRC_ROOT = next(
    (
        root
        for root in [os.path.abspath("..")] + sorted(glob.glob("/kaggle/input/*"))
        if os.path.isfile(os.path.join(root, "src", "tiling.py"))
    ),
    None,
)
if SRC_ROOT is None:
    raise FileNotFoundError("src/ not found; on Kaggle, attach the project's src folder as a dataset")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

In [2]:
"""Imports.

datetime, os, time    the record of when and how the models were made, paths, timing.
pandas                labels.
torch                 the GPU name for the record.
tqdm                  progress over seeds.
src.features          the packed set, and the code commit for the record.
src.mil               training, saving, loading and the ensemble prediction.
src.paths             folds.csv and where the heads are saved.
"""

import datetime
import os
import time

import pandas as pd
import torch
from tqdm.auto import tqdm

from src import mil
from src.features import (
    code_commit,  # the current commit, -dirty if code has changed: recorded with the models
    open_packed,  # the packed tissue set
)
from src.mil import (
    EPOCHS,  # 20 epochs, no early stopping
    HEAD_NEEDS_POSITIONS,  # which heads need tile positions (the RoPE head)
    SEEDS,  # 0 to 4: one model per seed
    fit_final,  # trains one model per seed on every slide, returns their weights
    load_bags,  # every slide's features into RAM
    load_ensemble,  # a saved head's models, ready to predict
    load_positions,  # every slide's tile grid positions
    predict_ensemble,  # one slide's grade from the averaged probabilities of a head's models
    save_ensemble,  # a head's models, saved together with their record
)
from src.paths import (
    DATA_DIR,  # folds.csv lives here locally
    find_input,  # finds folds.csv
    head_path,  # where a head's file is saved: data/models/<name>.pt
)

In [3]:
"""Settings and labels.

1. VERSION: the packed tissue set every result so far used.
2. HEADS and LOSS: the decision of 2026-11-19.
3. labels: every packed slide (the empty slide is not in the packed set).
4. RECORD: saved with each head, so a file always says what made it.
"""

VERSION = "uni2h-L1-224-t144-hsv-v1"
HEADS = ["abmil", "roformer"]
LOSS = "soft_labels"

packed = open_packed(VERSION)
folds = pd.read_csv(find_input("folds.csv", local_dir=DATA_DIR)).set_index("slide_id")
labels = folds.loc[packed.index["slide_id"]]
RECORD = {
    "feature_set": VERSION,
    "training_slides": len(labels),
    "epochs": EPOCHS,
    "settings": {"hidden": mil.HIDDEN, "attention": mil.ATTENTION, "dropout": mil.DROPOUT, "lr": mil.LR,
                 "weight_decay": mil.WEIGHT_DECAY, "heads": mil.HEADS, "mlp_ratio": mil.MLP_RATIO,
                 "soft_label_sigma": mil.SOFT_LABEL_SIGMA, "grid_cell": mil.GRID_CELL},
    "code_commit": code_commit(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
}
print(f"{len(labels):,} training slides; heads {HEADS}; loss {LOSS}; seeds {SEEDS}; commit {RECORD['code_commit']}")

folds.csv: /home/aztrek/projects/Prostate-Cancer-Grading/data/folds.csv
10,615 training slides; heads ['abmil', 'roformer']; loss soft_labels; seeds [0, 1, 2, 3, 4]; commit f3199a4-dirty


## Train and save

One model per seed on every slide, for each head; then each head's five models
are saved in one file. About 15 minutes for ABMIL and an hour for the RoPE
model.

In [4]:
"""Train each head on every slide once per seed, and save its models together."""

bags = load_bags(packed)
positions = load_positions(packed)
for head in HEADS:
    started = time.perf_counter()
    states = fit_final(bags, labels, head, LOSS, positions=positions, progress=tqdm)
    path = head_path(f"{head}_{LOSS}")
    info = {**RECORD, "created": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
            "training_minutes": round((time.perf_counter() - started) / 60, 1)}
    save_ensemble(path, head, LOSS, SEEDS, states, info)
    print(f"{head}: {len(states)} models in {info['training_minutes']} min -> {path} ({os.path.getsize(path) / 1e6:.1f} MB)")

abmil: 5 models in 14.4 min -> /home/aztrek/projects/Prostate-Cancer-Grading/data/models/abmil_soft_labels.pt (21.1 MB)
roformer: 5 models in 54.3 min -> /home/aztrek/projects/Prostate-Cancer-Grading/data/models/roformer_soft_labels.pt (84.1 MB)


## Do the saved files load and predict?

Each file is loaded back the way the submission will load it, and predicts the
reproduction test's four slides with tiles from their stored features. These
slides were in training, so the grades are a plumbing check, not a score.

In [5]:
"""Reload each head's file and predict four training slides from their stored features."""

check_ids = ["1836f6539ccc9e37d426603cc4526f8b", "4726ffb66a6516baff6de98df2b5a26b",
             "0240270d611683829ea5bee2a8c52eb0", "03b113d63b06f6740d82b4cfa1ee4a75"]
for head in HEADS:
    models, record = load_ensemble(head_path(f"{head}_{LOSS}"))
    print(f"{head}: {len(models)} models, loss {record['loss']}, seeds {record['seeds']}, "
          f"commit {record['info']['code_commit']}")
    for slide_id in check_ids:
        grade, probabilities = predict_ensemble(models, record["loss"], bags[slide_id],
                                                positions[slide_id] if HEAD_NEEDS_POSITIONS[head] else None)
        print(f"  {slide_id}: true {labels.loc[slide_id, 'isup_grade']}, predicted {grade}, "
              f"probabilities {probabilities.round(3).tolist()}")

abmil: 5 models, loss soft_labels, seeds [0, 1, 2, 3, 4], commit f3199a4-dirty
  1836f6539ccc9e37d426603cc4526f8b: true 1, predicted 1, probabilities [0.1589999943971634, 0.375, 0.33000001311302185, 0.12099999934434891, 0.014000000432133675, 0.0010000000474974513]
  4726ffb66a6516baff6de98df2b5a26b: true 2, predicted 2, probabilities [0.0820000022649765, 0.28600001335144043, 0.4050000011920929, 0.20000000298023224, 0.026000000536441803, 0.0010000000474974513]
  0240270d611683829ea5bee2a8c52eb0: true 3, predicted 3, probabilities [0.008999999612569809, 0.09399999678134918, 0.30000001192092896, 0.3799999952316284, 0.18000000715255737, 0.03700000047683716]
  03b113d63b06f6740d82b4cfa1ee4a75: true 4, predicted 4, probabilities [0.0010000000474974513, 0.012000000104308128, 0.07999999821186066, 0.27000001072883606, 0.40799999237060547, 0.2290000021457672]
roformer: 5 models, loss soft_labels, seeds [0, 1, 2, 3, 4], commit f3199a4-dirty
  1836f6539ccc9e37d426603cc4526f8b: true 1, predicted 1,

## Result

Run 2026-11-19 on the RTX 4060 PC. Both heads trained with soft labels on all
10,615 packed slides of `uni2h-L1-224-t144-hsv-v1`, seeds 0 to 4, and saved:

| Head | File | Size | Training |
|---|---|---|---|
| ABMIL | `data/models/abmil_soft_labels.pt` | 21.1 MB | 14.4 min |
| RoPE model | `data/models/roformer_soft_labels.pt` | 84.1 MB | 54.3 min |

Both files load back (weights only) and predict the four check slides with the
averaged probabilities of their five models. Those slides were in training, so
this checks the files, not the models. The record in each file names the commit
as `f3199a4-dirty`: the training code (`fit_final`, `save_ensemble`,
`load_ensemble`, `predict_ensemble`, `grid_positions`) was committed right after
this run.